1. Load image_manifest.csv and select images where split == "validation".
2. Use the finalized prompt and JSON Schema.
3. Save each Gemini prediction individually, with API error handling and automatic skipping of previously processed images.
4. Perform the same quantitative evaluations on all 61 validation images.
5. Generate final comparison tables for follicle-like region counts, sizes, and spatial agreement.

In [ ]:
from google.colab import drive
from pathlib import Path
import pandas as pd
import json

drive.mount("/content/drive")

PROJECT_DIR = Path(
    "/content/drive/MyDrive/ovarian_ultrasound_vlm"
)

MANIFEST_PATH = PROJECT_DIR / "image_manifest.csv"

manifest = pd.read_csv(MANIFEST_PATH)

assert len(manifest) == 301
assert manifest["case_id"].is_unique

print("Total dataset images:", len(manifest))

Mount Google Drive and load the dataset manifest for the main evaluation

In [ ]:
validation = (
    manifest[manifest["split"] == "validation"]
    .sort_values("case_id")
    .reset_index(drop=True)
)

assert len(validation) == 61
assert validation["case_id"].is_unique

# Make sure no pilot images are included
assert not validation["is_pilot"].astype(str).str.lower().eq("true").any()

# Verify that all images and pseudo-labels exist
for _, row in validation.iterrows():
    assert Path(row["image_path"]).exists()
    assert Path(row["pseudo_label_path"]).exists()

print("Validation images:", len(validation))

print("\nCategory distribution:")
print(validation["category"].value_counts().sort_index())

display(validation[["case_id", "category", "split"]].head(10))

Select the 61 validation cases from the original nnU-Net fold 0

In [ ]:
# Load the configuration saved during the pilot experiment.

CONFIG_PATH = (
    PROJECT_DIR / "config" / "gemini_pilot_config.json"
)

assert CONFIG_PATH.exists(), (
    f"Configuration not found: {CONFIG_PATH}"
)

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    config = json.load(f)

MODEL_ID = config["model_id"]
ANNOTATION_PROMPT = config["prompt"]
RESPONSE_SCHEMA = config["response_schema"]
TEMPERATURE = config["temperature"]

assert config["prompt_version"] == "v1"
assert len(config["pilot_case_ids"]) == 10
assert config["bbox_coordinate_range"] == [0, 1000]
assert TEMPERATURE == 0

print("Model:", MODEL_ID)
print("Prompt version:", config["prompt_version"])
print("Temperature:", TEMPERATURE)
print("Schema loaded:", bool(RESPONSE_SCHEMA))
print("Prompt length:", len(ANNOTATION_PROMPT))

Load and verify the frozen Gemini pilot configuration

In [ ]:
from google import genai
from google.colab import userdata
from google.genai import types
from pydantic import BaseModel, Field
from typing import List, Literal

# Use the same Pydantic schema as in the pilot experiment.
class FollicleRegion(BaseModel):
    bbox: List[int] = Field(
        description="Bounding box [xmin, ymin, xmax, ymax], normalized to 0-1000."
    )
    shape: Literal["round", "oval", "irregular", "uncertain"]
    appearance: str

class UltrasoundAnnotation(BaseModel):
    follicle_like_count: int
    regions: List[FollicleRegion]
    image_quality: Literal["good", "moderate", "poor"]
    uncertain: bool
    description: str

# Verify consistency with the saved pilot schema.
assert UltrasoundAnnotation.model_json_schema() == RESPONSE_SCHEMA, (
    "The current annotation schema differs from the pilot schema."
)

API_KEY = userdata.get("GEMINI_API_KEY")
if not API_KEY:
    raise ValueError("GEMINI_API_KEY is missing.")

client = genai.Client(api_key=API_KEY)

# Separate directory for the main experiment.
MAIN_PREDICTIONS_DIR = (
    PROJECT_DIR / "predictions" / "main_validation"
)
MAIN_PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)

print("Gemini client initialized.")
print("Selected model:", MODEL_ID)
print("Response schema verified.")
print("Output directory:", MAIN_PREDICTIONS_DIR)

Initialize the Gemini client and validate the saved response schema

In [ ]:
import json
import time
from datetime import datetime, timezone
from pathlib import Path
from PIL import Image

# Start with a small batch to verify the main pipeline.
MAX_NEW_REQUESTS = 5
REQUEST_DELAY_SECONDS = 6

successes = []
failures = []
skipped = []

for _, row in validation.iterrows():

    case_id = row["case_id"]
    output_path = MAIN_PREDICTIONS_DIR / f"{case_id}.json"

    # Skip already processed images.
    if output_path.exists():
        skipped.append(case_id)
        print(f"SKIP: {case_id}")
        continue

    # Limit new API requests per execution.
    if len(successes) + len(failures) >= MAX_NEW_REQUESTS:
        print("Batch limit reached.")
        break

    print(f"\nProcessing: {case_id}")

    try:
        with Image.open(row["image_path"]) as img:
            image = img.convert("RGB")

        start_time = time.monotonic()

        response = client.models.generate_content(
            model=MODEL_ID,
            contents=[image, ANNOTATION_PROMPT],
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                response_schema=UltrasoundAnnotation,
                temperature=TEMPERATURE
            )
        )

        elapsed = time.monotonic() - start_time

        annotation = UltrasoundAnnotation.model_validate_json(
            response.text
        )

        # Validate annotation consistency.
        assert annotation.follicle_like_count >= 0
        assert annotation.follicle_like_count == len(
            annotation.regions
        )

        for region in annotation.regions:
            assert len(region.bbox) == 4
            x1, y1, x2, y2 = region.bbox

            assert 0 <= x1 < x2 <= 1000
            assert 0 <= y1 < y2 <= 1000

        result = {
            "case_id": case_id,
            "model": MODEL_ID,
            "prompt_version": config["prompt_version"],
            "temperature": TEMPERATURE,
            "timestamp_utc": datetime.now(timezone.utc).isoformat(),
            "inference_seconds": round(elapsed, 3),
            "annotation": annotation.model_dump()
        }

        # Save the result immediately.
        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(
                result,
                f,
                indent=2,
                ensure_ascii=False
            )

        successes.append(case_id)

        print(
            f"SUCCESS: {case_id} | "
            f"Regions: {annotation.follicle_like_count} | "
            f"Time: {elapsed:.2f}s"
        )

        time.sleep(REQUEST_DELAY_SECONDS)

    except Exception as e:
        failures.append({
            "case_id": case_id,
            "error": str(e)
        })

        print(f"FAILED: {case_id}")
        print(str(e))

        error_text = str(e).lower()

        # Stop rather than retry when API limits are reached.
        if any(term in error_text for term in [
            "429",
            "resource_exhausted",
            "quota",
            "rate limit"
        ]):
            print("Stopping due to API limits.")
            break

print("\n--- Main inference batch summary ---")
print("New successful annotations:", len(successes))
print("Previously saved:", len(skipped))
print("Failures:", len(failures))

saved_count = sum(
    (MAIN_PREDICTIONS_DIR / f"{case_id}.json").exists()
    for case_id in validation["case_id"]
)

print(f"Total saved validation annotations: {saved_count}/61")

Run reproducible Gemini inference on the validation set with checkpointing and API error handling

In [ ]:
import pandas as pd
import json

records = []

for case_id in validation["case_id"]:

    path = MAIN_PREDICTIONS_DIR / f"{case_id}.json"

    if not path.exists():
        continue

    with open(path, "r", encoding="utf-8") as f:
        result = json.load(f)

    ann = result["annotation"]

    records.append({
        "case_id": case_id,
        "model": result["model"],
        "follicle_like_count": ann["follicle_like_count"],
        "image_quality": ann["image_quality"],
        "uncertain": ann["uncertain"],
        "inference_seconds": result.get("inference_seconds")
    })

main_summary = pd.DataFrame(records)

print(f"Completed: {len(main_summary)}/61")

if not main_summary.empty:
    print(
        "Total predicted regions:",
        main_summary["follicle_like_count"].sum()
    )

    display(main_summary)

Verify the saved validation annotations and summarize inference progress

In [ ]:
# Verify that all validation images have saved predictions.
assert len(main_summary) == 61
assert main_summary["case_id"].is_unique
assert main_summary["follicle_like_count"].sum() == 143

# Create evaluation directory.
MAIN_EVALUATION_DIR = PROJECT_DIR / "evaluation" / "main_validation"
MAIN_EVALUATION_DIR.mkdir(parents=True, exist_ok=True)

# Save summary.
summary_path = MAIN_EVALUATION_DIR / "gemini_main_summary.csv"

main_summary.to_csv(summary_path, index=False)

print("Main inference summary saved:", summary_path)
print("Number of images:", len(main_summary))
print(
    "Total predicted regions:",
    main_summary["follicle_like_count"].sum()
)

Save the completed Gemini validation inference summary

In [ ]:
import numpy as np
import pandas as pd
from PIL import Image
from skimage import measure

def measure_pseudo_label_regions(mask, case_id):
    """Measure all 8-connected foreground components without filtering."""

    labeled = measure.label(mask == 1, connectivity=2)
    records = []

    for region in measure.regionprops(labeled):
        min_row, min_col, max_row, max_col = region.bbox

        records.append({
            "case_id": case_id,
            "model": "BiomedParse",
            "region_id": region.label,
            "area_px2": region.area,
            "equivalent_diameter_px": region.equivalent_diameter_area,
            "major_axis_length_px": region.axis_major_length,
            "minor_axis_length_px": region.axis_minor_length,
            "bbox_width_px": max_col - min_col,
            "bbox_height_px": max_row - min_row,
            "bbox_xmin": min_col,
            "bbox_ymin": min_row,
            "bbox_xmax": max_col,
            "bbox_ymax": max_row
        })

    return records


# Process all 61 validation pseudo-labels
all_measurements = []
component_counts = {}

for _, row in validation.iterrows():
    case_id = row["case_id"]

    with Image.open(row["pseudo_label_path"]) as img:
        mask = np.array(img)

    measurements = measure_pseudo_label_regions(mask, case_id)

    all_measurements.extend(measurements)
    component_counts[case_id] = len(measurements)

    print(f"{case_id}: {len(measurements)} components")

biomedparse_measurements = pd.DataFrame(all_measurements)

print("\nValidation images:", len(component_counts))
print("Total BiomedParse components:", len(biomedparse_measurements))

Extract connected components and geometric measurements from validation pseudo-labels

In [ ]:
output_path = (
    MAIN_EVALUATION_DIR /
    "biomedparse_main_measurements.csv"
)

biomedparse_measurements.to_csv(output_path, index=False)

print("Measurements saved:", output_path)
print("Total regions:", len(biomedparse_measurements))

Save BiomedParse geometric measurements for the main validation experiment

In [ ]:
import json
import pandas as pd
from PIL import Image

gemini_records = []

for _, row in validation.iterrows():
    case_id = row["case_id"]

    prediction_path = (
        MAIN_PREDICTIONS_DIR / f"{case_id}.json"
    )

    with open(prediction_path, "r", encoding="utf-8") as f:
        result = json.load(f)

    annotation = result["annotation"]

    with Image.open(row["image_path"]) as img:
        width, height = img.size

    assert annotation["follicle_like_count"] == len(
        annotation["regions"]
    )

    for region_id, region in enumerate(
        annotation["regions"], start=1
    ):
        x1, y1, x2, y2 = region["bbox"]

        # Convert normalized coordinates to original pixels.
        xmin = x1 / 1000 * width
        ymin = y1 / 1000 * height
        xmax = x2 / 1000 * width
        ymax = y2 / 1000 * height

        assert 0 <= xmin < xmax <= width
        assert 0 <= ymin < ymax <= height

        bbox_width = xmax - xmin
        bbox_height = ymax - ymin

        gemini_records.append({
            "case_id": case_id,
            "model": result["model"],
            "region_id": region_id,
            "bbox_xmin": xmin,
            "bbox_ymin": ymin,
            "bbox_xmax": xmax,
            "bbox_ymax": ymax,
            "bbox_width_px": bbox_width,
            "bbox_height_px": bbox_height,
            "bbox_area_px2": bbox_width * bbox_height,
            "shape": region["shape"],
            "appearance": region["appearance"]
        })

gemini_measurements = pd.DataFrame(gemini_records)

assert len(gemini_measurements) == 143

print("Validation images:", len(validation))
print("Total Gemini regions:", len(gemini_measurements))

display(gemini_measurements.head(10))

Extract pixel-based bounding-box measurements from the saved Gemini validation annotations

In [ ]:
gemini_output_path = (
    MAIN_EVALUATION_DIR / "gemini_main_measurements.csv"
)

gemini_measurements.to_csv(
    gemini_output_path,
    index=False
)

print("Gemini measurements saved:", gemini_output_path)
print("Total regions:", len(gemini_measurements))

Save Gemini validation bounding-box measurements for quantitative comparison

In [ ]:
# Count connected components and predicted bounding boxes per image.
bp_counts = (
    biomedparse_measurements
    .groupby("case_id")
    .size()
    .rename("biomedparse_count")
)

gemini_counts = (
    gemini_measurements
    .groupby("case_id")
    .size()
    .rename("gemini_count")
)

# Include all 61 images, including zero-count predictions.
count_comparison = (
    validation[["case_id", "category"]]
    .set_index("case_id")
    .join(bp_counts)
    .join(gemini_counts)
    .fillna(0)
    .reset_index()
)

count_comparison["biomedparse_count"] = (
    count_comparison["biomedparse_count"].astype(int)
)

count_comparison["gemini_count"] = (
    count_comparison["gemini_count"].astype(int)
)

# Calculate count agreement.
count_comparison["absolute_error"] = (
    count_comparison["gemini_count"]
    - count_comparison["biomedparse_count"]
).abs()

count_mae = count_comparison["absolute_error"].mean()

assert len(count_comparison) == 61
assert count_comparison["biomedparse_count"].sum() == 79
assert count_comparison["gemini_count"].sum() == 143

print(f"Validation Count MAE: {count_mae:.3f}")

print("\nCount agreement by category:")
display(
    count_comparison.groupby("category")[
        "absolute_error"
    ].agg(["mean", "median", "count"])
)

display(count_comparison)

Compare region counts between Gemini and BiomedParse on the validation set

In [ ]:
count_path = (
    MAIN_EVALUATION_DIR / "main_count_comparison.csv"
)

count_comparison.to_csv(count_path, index=False)

print("Count comparison saved:", count_path)

Save the main validation count comparison for thesis analysis

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment

IOU_THRESHOLD = 0.5

def calculate_iou(box_a, box_b):
    x1 = max(box_a[0], box_b[0])
    y1 = max(box_a[1], box_b[1])
    x2 = min(box_a[2], box_b[2])
    y2 = min(box_a[3], box_b[3])

    intersection = max(0, x2 - x1) * max(0, y2 - y1)

    area_a = max(0, box_a[2] - box_a[0]) * max(
        0, box_a[3] - box_a[1]
    )
    area_b = max(0, box_b[2] - box_b[0]) * max(
        0, box_b[3] - box_b[1]
    )

    union = area_a + area_b - intersection
    return intersection / union if union > 0 else 0.0


def get_boxes(df):
    return df[
        ["bbox_xmin", "bbox_ymin", "bbox_xmax", "bbox_ymax"]
    ].to_numpy(dtype=float)


comparison_records = []
matched_records = []

for case_id in validation["case_id"]:

    bp_regions = biomedparse_measurements[
        biomedparse_measurements["case_id"] == case_id
    ].reset_index(drop=True)

    gem_regions = gemini_measurements[
        gemini_measurements["case_id"] == case_id
    ].reset_index(drop=True)

    bp_boxes = get_boxes(bp_regions)
    gem_boxes = get_boxes(gem_regions)

    n_bp = len(bp_boxes)
    n_gem = len(gem_boxes)

    iou_matrix = np.zeros((n_bp, n_gem))

    for i, bp_box in enumerate(bp_boxes):
        for j, gem_box in enumerate(gem_boxes):
            iou_matrix[i, j] = calculate_iou(
                bp_box, gem_box
            )

    valid_matches = []

    if n_bp > 0 and n_gem > 0:
        matching_score = (
            (iou_matrix >= IOU_THRESHOLD).astype(float) * 2
            + iou_matrix
        )

        row_ind, col_ind = linear_sum_assignment(
            matching_score,
            maximize=True
        )

        for i, j in zip(row_ind, col_ind):
            iou = float(iou_matrix[i, j])

            if iou >= IOU_THRESHOLD:
                valid_matches.append(iou)

                matched_records.append({
                    "case_id": case_id,
                    "biomedparse_region_id": int(
                        bp_regions.iloc[i]["region_id"]
                    ),
                    "gemini_region_id": int(
                        gem_regions.iloc[j]["region_id"]
                    ),
                    "bbox_iou": iou
                })

    tp = len(valid_matches)

    comparison_records.append({
        "case_id": case_id,
        "biomedparse_count": n_bp,
        "gemini_count": n_gem,
        "matched_regions": tp,
        "unmatched_gemini": n_gem - tp,
        "unmatched_biomedparse": n_bp - tp,
        "mean_matched_iou": (
            float(np.mean(valid_matches))
            if valid_matches else np.nan
        )
    })

spatial_comparison = pd.DataFrame(comparison_records)
matched_regions = pd.DataFrame(
    matched_records,
    columns=[
        "case_id",
        "biomedparse_region_id",
        "gemini_region_id",
        "bbox_iou"
    ]
)

display(spatial_comparison.head(10))

Evaluate spatial agreement between Gemini and BiomedParse bounding boxes

In [ ]:
tp = int(spatial_comparison["matched_regions"].sum())
fp = int(spatial_comparison["unmatched_gemini"].sum())
fn = int(spatial_comparison["unmatched_biomedparse"].sum())

precision = tp / (tp + fp) if tp + fp else 0
recall = tp / (tp + fn) if tp + fn else 0

f1 = (
    2 * precision * recall / (precision + recall)
    if precision + recall else 0
)

mean_matched_iou = (
    matched_regions["bbox_iou"].mean()
    if not matched_regions.empty else np.nan
)

assert tp + fp == 143
assert tp + fn == 79

print("Main validation spatial agreement")
print("---------------------------------")
print(f"Count MAE: {count_mae:.3f}")
print(f"IoU threshold: {IOU_THRESHOLD}")
print(f"Matched regions: {tp}")
print(f"Unmatched Gemini regions: {fp}")
print(f"Unmatched BiomedParse regions: {fn}")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1-score: {f1:.3f}")
print(f"Mean matched IoU: {mean_matched_iou:.3f}")

Calculate aggregate spatial agreement metrics for all 61 validation images

In [ ]:
spatial_path = MAIN_EVALUATION_DIR / "main_spatial_comparison.csv"
matches_path = MAIN_EVALUATION_DIR / "main_matched_regions.csv"

spatial_comparison.to_csv(spatial_path, index=False)
matched_regions.to_csv(matches_path, index=False)

print("Saved:", spatial_path)
print("Saved:", matches_path)

Save spatial agreement metrics and matched region pairs